In [5]:
import torch
import numpy as np
from tqdm import tqdm
import scipy

from utils import *

d1 = 100000
d2 = 1000
r = 10
p = 0.001
ob = 5
sample = "iid"
if torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'

# dataset
dataset = "syn"
#print(dataset)

total_t = 5
total_cov_err_list = []
total_cov_err_mask_list = []
total_T_p_err_list = []
total_T_p_err_mask_list = []
total_T_err_list = []
total_T_err_mask_list = []
total_X_err_list = []
total_X_err_mask_list = []

for t in range(total_t):
    M = load_data_syn(r, d1, d2, device)
    
    T_p_list = []
    T_list = []
    T_p_err_list = []
    T_err_list = []

    if sample == "iid":
        # IID sample
        observed_M, masks = get_uniform_masks(M, p)
    else:
        # few entry sample
        observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), ob)
        p = ob/d2
        observed_M = torch.from_numpy(observed_M).float().to(device)
        masks = torch.from_numpy(masks).to(device)

    # observed second-moment matrix
    MTM = M.T @ M
    cov_observe_M =  observed_M.T @ observed_M
    T_masks = 1 * (cov_observe_M!=0)

    # True probability weighting
    diag_cov = torch.diag( torch.diag(cov_observe_M) )
    T_p = ((1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov))

    # Inverse estimated probability weighting in matrix
    cov_observe_count = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
    cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
    p_hat_matrix = cov_observe_count/d1

    T = cov_observe_M / p_hat_matrix
    T_err_list.append((torch.norm(T*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks)).item())

    cov_err_mask = torch.norm(cov_observe_M*T_masks - MTM*T_masks, 'fro') / torch.norm(MTM*T_masks, 'fro')
    cov_err = torch.norm(cov_observe_M - MTM, 'fro') / torch.norm(MTM, 'fro')
    total_cov_err_list.append(cov_err.item())
    total_cov_err_mask_list.append(cov_err_mask.item())

    T_p_err_mask = torch.norm(T_p*T_masks - MTM*T_masks, 'fro') / torch.norm(MTM*T_masks, 'fro')
    T_p_err = torch.norm(T_p - MTM, 'fro') / torch.norm(MTM, 'fro')
    total_T_p_err_list.append(T_p_err.item())
    total_T_p_err_mask_list.append(T_p_err_mask.item())

    T_err_mask = torch.norm(T*T_masks - MTM*T_masks, 'fro') / torch.norm(MTM*T_masks, 'fro')
    T_err = torch.norm(T - MTM, 'fro') / torch.norm(MTM, 'fro')
    total_T_err_list.append(T_err.item())
    total_T_err_mask_list.append(T_err_mask.item())

    # impute missing values from rank-r SVD corresponding to masks
    use_power_method = False
    train_losses = []
    err_estimates = []

    epochs = 30
    tol = 1e-7
    lr = 0.1
    X = T
    
    #T_masks = 1 * (T != 0)
    print(T_masks.sum())
    loop = tqdm(range(epochs))
    
    for i in loop:
        U, D, Vt = torch.linalg.svd(X)
        D[r:] = 0
        X_update = U @ torch.diag(D) @ Vt

        X = X * T_masks + X_update * (1 - T_masks)
        #X = X * (1-lr) + X_update * lr
        err = MTM - X
        loss = (err**2).mean()
        train_losses.append(loss.item())
        relative_err = torch.norm(err, 'fro') / torch.norm(MTM, 'fro')
        if i > 3:
            if err_estimates[-1] - err_estimates[-2] < tol:
                break
        loop.set_description(f"relative err: {relative_err:.7f}")
        err_estimates.append(relative_err.item())
        #print(relative_err)

    X_err_mask = torch.norm(X*T_masks - MTM*T_masks, 'fro') / torch.norm(MTM*T_masks, 'fro')
    X_err = torch.norm(X - MTM, 'fro') / torch.norm(MTM, 'fro')
    total_X_err_list.append(X_err.item())
    total_X_err_mask_list.append(X_err_mask.item())

print(f"cov_err: {np.mean(total_cov_err_list)} +- {np.std(total_cov_err_list)}")
print(f"cov_err_mask: {np.mean(total_cov_err_mask_list)} +- {np.std(total_cov_err_mask_list)}")
print(f"T_p_err: {np.mean(total_T_p_err_list)} +- {np.std(total_T_p_err_list)}")
print(f"T_p_err_mask: {np.mean(total_T_p_err_mask_list)} +- {np.std(total_T_p_err_mask_list)}")
print(f"T_err: {np.mean(total_T_err_list)} +- {np.std(total_T_err_list)}")
print(f"T_err_mask: {np.mean(total_T_err_mask_list)} +- {np.std(total_T_err_mask_list)}")
print(f"X_err: {np.mean(total_X_err_list)} +- {np.std(total_X_err_list)}")
print(f"X_err_mask: {np.mean(total_X_err_mask_list)} +- {np.std(total_X_err_mask_list)}")


tensor(96122, device='cuda:0')


relative err: 0.6810255:  13%|█▎        | 4/30 [00:01<00:06,  3.95it/s]


tensor(94906, device='cuda:0')


relative err: 0.6849111:  13%|█▎        | 4/30 [00:01<00:07,  3.51it/s]


tensor(97872, device='cuda:0')


...

tensor(95786, device='cuda:0')


relative err: 0.6825677:  13%|█▎        | 4/30 [00:01<00:07,  3.51it/s]


tensor(95596, device='cuda:0')


relative err: 0.6826978:  13%|█▎        | 4/30 [00:01<00:07,  3.66it/s]

cov_err: 0.9999979734420776 +- 0.0
cov_err_mask: 0.9999791860580445 +- 6.951036327892901e-08
T_p_err: 3.1670993328094483 +- 0.014672790052621516
T_p_err_mask: 9.747283744812012 +- 0.010858356779905933
T_err: 0.9510693907737732 +- 0.0005188710685267216
T_err_mask: 0.07871772050857544 +- 0.00036603096850547346
X_err: 0.6361795425415039 +- 0.0031755399562351573
X_err_mask: 0.07871772050857544 +- 0.00036603096850547346
